# Leson 10 - AI Agents for Production

For dis leson you go learn **production patterns** for AI agents wey dey use the Microsoft Agent Framework (Python). We go cover:

- **Observability** — how to add timing and logging to agent interactions
- **Evaluation** — how to use an evaluator agent to score di quality of responses
- **Cost management** — strategies for token optimization and model selection

Di scenario na a **travel agent** wey dey help users plan trips, with monitoring and evaluation wey dey layered on top.


## How to set am


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Wetin to Consider for Production

To move AI agents from prototypes to production, you need to pay careful attention to three pillars:

1. **Observability** — You need make you fit see wetin the agent dey do, how long e dey take, and which tools e dey call. If no get tracing and logging, to debug production wahala near impossible.

2. **Evaluation** — Automated quality checks dey make sure say the agent responses remain correct, complete, and helpful as time dey go. One evaluator agent fit score the responses against the defined criteria.

3. **Cost Management** — How many tokens you use dey directly affect cost. Strategies like prompt optimization, model selection, and caching dey help keep expenses under control without sacrificing quality.


## How we dey build Observable Agent

We go define travel tools and wrap the agent call with timing make we fit monitor latency. For production, you go integrate am with OpenTelemetry or another tracing backend wey similar.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Patterns wey dem dey use for Evaluation

One common production pattern na to use a second agent as an **person wey dey evaluate**. Di person wey dey evaluate go score di primary agent response according to criteria wey dem don set before, like whether e complete, correct, and whether e dey helpful.

Dis dey enable:
- Automated quality gates wey dey check before responses reach users
- Regression detection wen prompts or models change
- Continuous monitoring of how agent dey perform over time


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## How to Manage Cost

To control cost na important thing for AI agents wey dey run for production. Below na the main strategies:

| Strategy | Description |
|---|---|
| **Prompt optimization** | Make system instructions short and clear. Comot extra context wey no need so e go reduce input tokens. |
| **Model selection** | Use smaller, cheaper models (e.g. GPT-4o-mini) for simple tasks like classification or extraction, and make big models dey handle complex reasoning. |
| **Caching** | Cache tool results and frequent queries so you no go dey make the same API calls again. |
| **Token budgets** | Set `max_tokens` limits so responses no go too long without you expect am. |
| **Batching** | Group many user queries into one API call where e possible. |

For real use, tiered approach dey work well: send straightforward requests to a fast, cheap model and only escalate complex queries to a more capable one.


## Summary

For this lesson you don learn how to:

1. **Make agent interactions dey observable** with timing and logging, wey go set the groundwork for tracing and monitoring.
2. **Check agent responses** automatically using an evaluator agent wey dey score if dem complete, correct, and helpful.
3. **Manage cost dem** by optimizing prompts, choosing models, using caching, and setting token budgets.

These production patterns dey help make sure say your AI agents dey reliable, measurable, and cost-effective when una scale dem.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Disclaimer:
Dis document dem translate using AI translation service Co-op Translator (https://github.com/Azure/co-op-translator). Even though we dey try make am correct, abeg note say automated translation fit get errors or mistakes. Di original document for im own language na di authoritative source. For any important mata, make you use professional human translation. We no dey responsible for any misunderstanding or wrong interpretation wey fit come from this translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
